# 第 1 周末练习 —— QF-Test 手册技术问答

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：关于 QF-Test 自动化测试手册的技术问题
- **输出**：基于手册上下文的清晰解释（变量清单、如何取回 SQL 结果等）
- **额外要求**：用**流式（streaming）**一边生成一边用 Markdown 刷新显示

这是你在课程期间自己也能天天用的工具：把「某产品文档 + 具体操作问题」丢给模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 给出手册链接与角色；user 放具体问题 |
| 流式输出 `stream=True` | 边收 `delta.content` 边 `update_display` |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），经 OpenAI 兼容 `/v1` 接口 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地路径还需本机 Ollama 在跑且已拉取 `llama3.2`
3. 在「提问」单元格改写 `question`（或 `system_prompt`），再分别跑 GPT 与 Llama 两格做对比


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：调用云端（或 Ollama 兼容）Chat Completions API
from openai import OpenAI


In [10]:
# ========== 常量 + 默认客户端：模型名集中写一处 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'
# 创建默认 OpenAI 客户端：密钥通常来自环境变量 OPENAI_API_KEY（后面 load_dotenv 后再用）
openai = OpenAI()


In [12]:
# ========== 环境配置：把 .env 读进进程 ==========

# override=True：.env 中的值覆盖已有环境变量，避免本机残留旧 Key
load_dotenv(override=True)
# 取出 OPENAI_API_KEY（本练习用常见名字 OPENAI_API_KEY；读取后若后面要显式传 api_key 可复用）
api_key = os.getenv('OPENAI_API_KEY')


In [13]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# system_prompt：角色与上下文（发给模型的指令，保留英文，改译会改变行为）
# 这里把 QF-Test 官方手册 URL 写进 system，让模型按「手册助手」回答
system_prompt = """
You are provided with the link of a website: https://www.qftest.com/doc/manual/en/manual.html.
It contains a manual of application used for automated testing.
You are able to answer questions about using the application based on the manual.
"""
# question：真正的用户问题（保留英文）；练习时可换成你自己手册里的另一段流程
question = """
Please explain how to use a procedure qfs.database.executeSelectStatement in QF-test.
1. List the variables needed to use this procedure.
2. Explain how to return the result of SQL query used in this procedure, which has one column and several rows?
"""


In [17]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 调用 Chat Completions；stream=True 表示服务端持续推送增量，而不是等整段答完
stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            # system：手册助手角色与文档链接
            {"role": "system", "content": system_prompt},
            # user：具体技术问题
            {"role": "user", "content": question}
        ],
        stream=True
    )
# 累积完整回答文本，供每次刷新 Markdown 使用
response = ""
# 先放一个空的 Markdown 占位，拿到 display_id，后面用同一 id 原地更新（流式「打字机」效果）
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk：每个 chunk 可能带一小段 delta.content
for chunk in stream:
    # or ''：有的 chunk 没有 content（例如结束标记），避免把 None 拼进字符串
    response += chunk.choices[0].delta.content or ''
    # 用同一 display_id 刷新整段 Markdown，笔记本里会看到文字逐渐变长
    update_display(Markdown(response), display_id=display_handle.display_id)


To use the procedure `qfs.database.executeSelectStatement` in QF-test, follow these steps:

### 1. List the Variables Needed
- **Connection String**: A string that contains the database connection information. It typically includes the database type, server address, database name, user credentials, etc.
- **SQL Statement**: The SQL query that you want to execute (e.g., `SELECT * FROM table_name`).
- **Result Variable**: A variable where the result of the executed SQL query will be stored. This variable should be compatible with the structure of the expected result (for example, a data table or list).

### 2. Returning the Result of the SQL Query
To return the result of an SQL query that outputs one column with several rows, you can follow these steps:

- Execute the `qfs.database.executeSelectStatement` procedure using the appropriate connection string and SQL statement.
- Store the result in the designated result variable.
- To retrieve the values, you can iterate over the rows of the result set stored in the result variable. 

Here is a simple outline on how to retrieve the values:
```plaintext
// 1. Declare variables
String connectionString = "Your_Connection_String";
String sqlStatement = "SELECT your_column FROM your_table;";
List<String> resultList;

// 2. Execute the SQL statement
resultList = qfs.database.executeSelectStatement(connectionString, sqlStatement);

// 3. Process the results
for (String value : resultList) {
    print(value); // or store in another variable as needed
}
```
Make sure to handle any exceptions or errors that may occur during the execution of the SQL statement or when processing the results.

In [19]:
# ========== 路径 B：用本地 Llama 3.2（Ollama OpenAI 兼容口）流式回答 ==========

# Ollama 的 OpenAI 兼容基址：/v1，这样可以复用同一个 OpenAI SDK
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 指向本地 Ollama；api_key 对本地常可填任意非空字符串（这里用 'ollama'）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 同一套 messages / stream=True，只是客户端与 model 换成本地 Llama
stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
# 与路径 A 相同的累积 + Markdown 刷新逻辑，便于对比云端 vs 本地
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


According to the manual of QF-test, `qfs.database.executeSelectStatement` is a procedure that executes a SELECT statement on a database.

**Variables needed:**

1. `select_sql`: The SQL query string.
2. `[optional]` `$timeout`: The maximum amount of time in milliseconds it can take for the execution to complete (default 5000).
3. `[optional]` `$output_file`: The file where the output will be saved (can be used if an error occurs).

**Usage and returning result:**

1. Call `qfs.database.executeSelectStatement` procedure with your SQL query string as the first argument.
2. If the procedure is successful, it will return a table containing one column (the results of the SQL query) and multiple rows.

To access the returned table, you can use the `$q` object to get its columns:

```qfm
qfs.database.executeSelectStatement("SELECT * FROM my_table WHERE condition");

$columns =
[
  "$col1" : "Column_1",
];

$table = $q->getTable(); // gets the result of qfs.database.executeSelectStatement

if ($table->hasRecordByIndex(0)) {
  record = array();
  record[] = "Row Data"; // first column
  print(record[0]); // prints the first row data
}
```

In your case, since you need to access a column with multiple values:

```qfm
qfs.database.executeSelectStatement("SELECT my_column FROM my_table");

// assuming the variable 'my_row_data' holds the data of one row,
$q->getFirstCol() // gets the first column (data type: str)
my_row_data = $q->getValue(); // or $q->asList();
```

These commands allow you to iterate over and access each row in the result.

However, the best approach will depend on your concrete use case.